# பாடம் 18 ( தொடர்ச்சியானது): ஒரு * மனிதர் * அனுமதி அளித்ததை நிரூபிக்கும் ரசீதுக்கள்

பாடம் **எஜெண்ட்** என்ன செய்தார் மற்றும் **கேட்** என்ன தீர்மானித்தது என்பதை நிரூபிக்கிறது. இந்த நோட்புக் காணாமல் போன பாதியைச் சேர்க்கிறது: ஒரு **பெயரிடப்பட்ட மனிதர்** **கானொணிக்கான** செயலை அங்கீகரித்தார் என்பதற்கான சான்று — முழு கானொணிக செயல்கூற்று மீதான தனித்துவமான, மனிதன் கொண்ட கையொப்பம், ஆன்லைனிலிருந்து வெறுமனே சரிபார்க்கப்பட்டது.

இங்கு இரு ஃபலக்ட்கள் பாடத்தின் ரசீதங்களின் **அதே உறை உருவத்தை** பயன்படுத்துகின்றன: ஒரு மேற்கோள் வகையை கொண்ட ஒரு பகுப்பு, Ed25519 மூலம் கையொப்பமிடப்பட்டுள்ளது, கானொணிக்கான செல்லுபடியான பகுப்பின் SHA-256 மீது, ஒரு கட்டமைக்கப்பட்ட `signature` பொருளுடன் இணைக்கப்பட்டது (மேலும் கையொப்பமிடப்பட்ட பைட்களில் இருந்து நீக்கப்பட்டது). அங்கீகார ரசீது ஒரு புதிய `type` (`human.approval.v1`), செயல் வகையின் அருகில் இருக்கிறது, ஆகவே ஒன்றை `verify_chain` கருத்து மூலம் இரு வகை ஃபலக்ட்களையும் ஒரே குறியீடு பாதையில் பராமரிக்கலாம், இது நீங்கள் முதன்மை நோட்புக்கில் உருவாக்கியது. இது பாடம் பின் தொடரும் Internet-Draft இல் கூட்டு கையொப்ப பாதையின் வடிவமும் ஆகும் (draft-farley-acta-signed-receipts).

முதன்மை நோட்புக்கின் டெமோ சரிபார்ப்பாளரைவிட ஒரு நோக்கமுள்ள மேம்பாடு: இங்கு சரிபார்ப்பாளர் `signature.key_id` ஐ ரசீதங்களில் உள்ள பொதுக் கீ பற்றி நம்புவதைவிட **நிராகரிக்கப்பட்ட விசை பதிவேடு** வைக்கப்பட்ட விசையுடன் ஒப்பிடுகிறது. அதுவே பாடத்தின் சொந்த சரிபார்ப்புக் குறிப்பான பிறக்குறுவையில் பரிந்துரைக்கப்படும் உற்பத்தி நிலை ("சரிபார்ப்புக் பொதுக் கீயை வெளியிடுக"), இது மோசடி செய்வதை மறுப்பாக மாற்றுகிறது, உங்கள் சொந்த விசையைக் கொண்டு தாண்டிவிடுவது биш.

இந்த நோட்புக் கற்பிக்கும் விதி: **ஒரு கையொப்பமிடப்பட்ட அங்கீகம் தானாக அதிகாரமாகாது.** அங்கீகார ரசீது மற்றும் செயல் ரசீது அநுபவ நேரத்தில் ஒரே கானொணிக்கான செயலுடன் இன்னும் பிணைக்கப்பட்டிருந்தால் மட்டுமே அதிகாரம் இருக்கும், அதில் மார்க்க வேர்ஷன், கீ மற்றும் காலாவதி இன்னும் செல்லுபடியாகும், மற்றும் ஏற்கனவே பயன்படுத்தப்படாத அங்கீகம். ஒவ்வொரு தோல்வியும் **தனித்துவ காரணம்** உடன் மறுக்கும்; ஆகையால் *அதிகாரம் பழையதாயிற்று* என்பதையும் *செயல்படுத்திய செயல்களில் மாற்றம் உள்ளது* என்பதையும் நீங்கள் தனித்து சொல்ல முடியும்.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## சரியான செயல்பாடு

ஒப்புதலுக்கான அலகு என்பது **காணொளி செயல்பாட்டு பொருள்** — "ஒப்புதல் பணத்தை திரும்பப்பெறு" போன்ற தெளிவற்ற லேபிள் இல்லாமல், நுண்ணறிந்த, முழுமையாக குறிப்பிடப்பட்ட செயல்பாடே ஆகும். முழு பொருளை கையெழுத்திடுதல் (மற்றும் அதிலிருந்து டைஜஸ்டை உருவாக்குதல்) தான் மனிதன் இதை மட்டும் ஒப்புதலிட்டார் என்பது பின்னர் நிரூபிக்க உதவுகிறது.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ஒரு லிபைடு, இரண்டு அதிகாரிகள்

ஒவ்வொரு ரசீதும் பாடத்தின் லிபைடு ஆகும்: ஒரு சமபில் பயலோடு `type` புலம், கூடுதலாக ஒரு `signature` பொருள் (`alg`, `sig`, `key_id`) யும், இது கையொப்பமிடப்பட்ட பைட்டுகளின் பகுதியாக இல்லை. `verify_envelope` இரு ரசீது வகைகளுக்கும் பொதுவான அமைப்பு + கையொப்ப சரிபார்ப்பு; எது **பின்னிய திறவுகோல் பதிவேட்டை** அது `signature.key_id` மீது தீர்மானிக்கிறது என்பது அதிகாரிகளை பிரித்துக் கொள்கிறது:

- **அங்கீகார ரசீது** (`human.approval.v1`) — பெயரிடப்பட்ட அங்கீகாரக்காரர், முழு canonical செயல்பாடு **மற்றும் அதன் சுருக்கம்**, `policy_version`, வெளியீடு + காலாவதி நேரங்கள். ஒரே முறையான பயன்படுத்தலை சங்கிலி நிலை திருத்துகிறது.
- **செயல் ரசீது** (`agent.action.v1`) — முகவரின் அடையாளம், `run_id`, அதே canonical செயல்பாட்டின் **சுருக்கம்**, செயல்படுத்தலின் முடிவு + நேரம், மற்றும் `parent_approval_ref`: அங்கீகாரத்தின் `receipt_hash`, பாடத்தின் சங்கிலியில் `previous_receipt_hash` ஆக இருப்பது போலவே.

பொதுவான `action_digest` புலம் இணைப்பின் அடிப்படையாகும். `key_id` கையொப்ப பொருளில் மட்டுமே தேடல் குறிப்பாக இருக்கின்றது: அதை வேறு பின்னிய திறவுகோலுக்கு மாற்றுதல் கையொப்ப சரிபார்ப்பை தோல்வியடையச் செய்கிறது, ஆகவே அது எதையும் தருவதில்லை.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: கட்டமைப்பு எங்கே உண்மையில் முடிவடைகிறது

`verify_chain` இரண்டு கையொப்பச் சரிபார்ப்புகளுக்கு மேலான வசதிப் பொருத்தவில்லை. இது பகிரப்பட்ட பொதுவான `action_digest`, அனுமதிக்கும் கொள்கை/தாவல்/காலாவதி **தற்சமயம்** மற்றும் அனுமதியின் **ஒரு முறை உபயோகப்படுத்தல்** ஆகியவை ஒன்றாக, செயல் *இப்போதே* நிறைவேற்றப்படுவதற்கு எதிராக சரிபார்க்கப்படும் ஒரே இடம் ஆகும்.

ஒவ்வொரு தோல்வியும் **பிரத்தியேக காரணத்தோடு** நிராகரிக்கிறது, ஆகையால் நிராகரிப்பை படிப்பவர் அதிகாரம் காலஞ்சென்றதா (கொள்கை மாற்றப்பட்டது, தாவல் சுழற்சி செய்யப்பட்டு விட்டது, அனுமதி காலாவதியாகி விட்டது, அனுமதி பயன்படுத்தப்பட்டது) அல்லது நிறைவேற்றப்படும் செயல் இன்னும் செல்லுபடியான அனுமதிக்கு கீழேயிருந்தாலும் (டைகெஸ்ட் மாற்றம்) மாற்றப்பட்டதா என்பதைச் சொல்ல முடியும்.


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## பைண்டிங் பிடிக்கும் பொருள்

கீழே உள்ள ஒவ்வொரு வழியும் **துண்டிக்கப்பட்டு** ஒரு **விதிவேறான காரணத்தால்** தோல்வியடைகிறது. முதல் பகுதி பாரம்பரிய தொகுப்பு (தவிர்க்க வைக்கப்பட்டவை, குழப்பப்பட்டப் பிரதிநிதி, மறுசுழற்சி, எந்த அதிகாரத்திலும் கூட்டு பதிவு, தவறான உள்ளீடு). இரண்டாவது பகுதி என்பது சொத்தாக உண்மையாக மாற்றும் ஜோடி:

- **பழுப்ப worn அதிகாரம்** — கையெழுத்து இன்னும் செல்லுபடியாகும், ஆனால் கொள்கை பதிப்பு நகர்ந்துவிட்டது, அங்கீகார விசை பின்தங்கப்பட்ட பதிவிலிருந்து மாற்றப்பட்டு விட்டது அல்லது அங்கீகாரம் நிறைவேற்றக்கான முன்பே காலாவதி ஆனது;
- **தொகுப்பு மாற்றம்** — செல்லுபடியாக கையாளப்பட்ட கிரியைக் கையெழுத்திடப்பட்ட ரசீதின் `parent_approval_ref` *உண்மையான* அங்கீகாரம் நோக்குகிறது, ஆனால் அந்த அங்கீகாரம் கொண்ட canonical கிரியைக் தொகுப்பு செயலாக்கப்படுகிற கிரியைக்கோ பொருந்தவில்லை.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## இது என்ன சான்று தருகிறது — மற்றும் தருவதில்லை

**சான்று தருகிறது:** ஒரு பெயரிடப்பட்ட மனிதர் *இந்தத் துல்லியமான canonical செயலை* (முழுமையான செயல் + ஹாஷ், ஒரு பின் பதிவு செய்யப்பட்ட பதிவகத்திலிருந்து தீர்க்கப்பட்ட விசையுடன் கையொப்பமிடப்பட்டது) ஒப்புக் கொண்டார், மற்றும் பிரதிநிதி *அதுவே அந்த ஒப்புக் கொண்ட செயல்பாடை* (அதே ஹாஷ், `receipt_hash` மூலம் ஒப்புதலுடன் கட்டுப்படுத்தப்பட்டது, பாடத்தின் சொந்த சங்க திட்டம்) செயல்படுத்தியது — ஒப்புதலின் கொள்கை பதிப்பு, விசை, மற்றும் காலவரையறு இழக்காமலேயே, ஒருமுறை மட்டும். எந்த ஒரு பக்கம் மாறினாலும் சங்கம் மூடப்படும், மறுப்பு காரணம் **எந்த** சொத்து உடைந்தது என்று சொல்லும்: பழைய அதிகாரம் அல்லது மாறிய செயல்பாடு.

**சான்று தருவதில்லை:** அந்த ஒப்புதல் UI மனிதருக்கு அவர்கள் கையொப்பமிடுவதாக நினைத்ததை காட்டியது (WYSIWYS தனிப்பட்ட சிக்கல்), விசை திரும்பவும் திரும்பவும் அல்லது திருடப்பட்டதா அல்லது அதை மாற்றுவதற்கு முன் பாதுகாக்கப்பட்டதா, அல்லது பிற விளைவுகள் செயல்பாட்டுடன் பொருந்தினவா என்பதைக் காட்டாது. கையொப்பமிடப்பட்டது என்றால் அங்கீகாரம் கிடைத்தது என அர்த்தமில்லை: பழைய கொள்கை மீது செல்லுபடியாகும் கையொப்பம், மாற்றப்பட்ட விசை, காலாவதியான சாளரம், அல்லது வேறுபட்ட ஹாஷ் இங்கு எந்த அங்கீகாரமும் தராது.

இரண்டு ரசீது வகைகளும் பாடத்தின் பரிசோதனை படுக்கையை மற்றும் ஒரே `verify_chain` குறியீட்டு பாதையை நோக்கமாகப் பகிர்ந்துகொள்கின்றன: முக்கிய நோட்புக் ஆன செயல் ரசீதுகள் binding தான் மனிதரின் ஒப்புதலை சரிபார்க்கும் குறியீடு. ஒரே சரிபார்ப்பாளர் ஒப்பந்தம், தனித்தனியான பின் அதிகாரிகள், canonical செயல் ஹாஷ் மற்றும் வேறெதுவும் இல்லாமல் இணைக்கப்பட்டுள்ளன.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
